# Aggregation and Grouping

The analytics team does not want individual rows. They want numbers. How
many tracks per genre? Who are the top artists by total listening time?

These are aggregation questions. We need to collapse many rows into summary
statistics, and SQL has powerful tools for exactly this.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
%pip install -q jupysql
%load_ext sql

In [ ]:
%sql sqlite:///../data/wavelength.sqlite

## Basic aggregates

The five fundamental aggregate functions are COUNT, SUM, AVG, MIN, and MAX.
Each takes a set of values and returns a single number.

`COUNT(*)` counts every row regardless of NULLs. `COUNT(column)` excludes
NULLs -- the difference between them is a handy data quality check.

In [ ]:
%%sql
SELECT
    COUNT(*) AS total_tracks,
    COUNT(play_count) AS tracks_with_play_count,
    SUM(play_count) AS total_plays,
    ROUND(AVG(play_count), 0) AS average_plays,
    MIN(play_count) AS min_plays,
    MAX(play_count) AS max_plays
FROM tracks;

## GROUP BY: aggregating per category

A single total is useful, but the analytics team usually wants breakdowns.
"How many tracks per genre?" is one number *per genre*. GROUP BY splits
the rows into groups and applies the aggregate to each one separately.

In [ ]:
%%sql
SELECT
    genre,
    COUNT(*) AS track_count,
    ROUND(AVG(play_count), 0) AS avg_plays
FROM tracks
WHERE play_count IS NOT NULL
GROUP BY genre
ORDER BY avg_plays DESC;

### The GROUP BY rule

There is one strict rule: **every column in SELECT must either appear in
GROUP BY or be inside an aggregate function.** If you group by genre, there
is one output row per genre -- SQL cannot pick an arbitrary track title to
put in that row. SQLite is lenient about this, but other databases will
throw an error. Do not rely on the lenient behaviour.

In [ ]:
%%sql
-- BAD PRACTICE: title is not aggregated or grouped
SELECT genre, title, COUNT(*) AS track_count
FROM tracks
GROUP BY genre
LIMIT 5;

You can group by multiple columns. This gives one row per artist-genre
combination.

In [ ]:
%%sql
SELECT
    a.name AS artist,
    t.genre,
    COUNT(*) AS track_count
FROM tracks t
INNER JOIN artists a ON t.artist_id = a.artist_id
GROUP BY a.name, t.genre
ORDER BY track_count DESC
LIMIT 15;

## HAVING vs WHERE

"Which genres have more than 150 tracks?" Your first instinct might be
WHERE, but that fails because SQL follows a specific execution order:

1. **FROM** -- 2. **WHERE** -- 3. **GROUP BY** -- 4. **HAVING** --
5. **SELECT** -- 6. **ORDER BY** -- 7. **LIMIT**

WHERE runs at step 2, before grouping. The count does not exist yet.
HAVING runs at step 4, after groups are formed. **WHERE filters rows;
HAVING filters groups.**

In [ ]:
%%sql
SELECT genre, COUNT(*) AS track_count
FROM tracks
GROUP BY genre
HAVING COUNT(*) > 150
ORDER BY track_count DESC;

You can use both in the same query. WHERE reduces rows before grouping;
HAVING reduces groups after.

In [ ]:
%%sql
SELECT genre, COUNT(*) AS track_count, ROUND(AVG(play_count), 0) AS avg_plays
FROM tracks
WHERE play_count IS NOT NULL
GROUP BY genre
HAVING COUNT(*) > 100
ORDER BY avg_plays DESC;

## COALESCE: replacing NULLs

`COALESCE` takes a list of values and returns the first non-NULL one. It is
useful when you want to treat missing data as zero rather than ignoring it.

In [ ]:
%%sql
SELECT
    genre,
    SUM(play_count) AS total_excluding_null,
    SUM(COALESCE(play_count, 0)) AS total_null_as_zero,
    COUNT(*) AS all_tracks,
    COUNT(play_count) AS tracks_with_plays
FROM tracks
GROUP BY genre
ORDER BY genre;

The SUM values are the same here, because SUM ignores NULLs anyway. But
the distinction matters for AVG: `AVG(play_count)` divides by non-NULL
count, while `AVG(COALESCE(play_count, 0))` divides by total count.

## Combining aggregation with JOINs

The real power shows when you combine aggregation with joins. Top 10
artists by total listening time:

In [ ]:
%%sql
SELECT
    a.name AS artist,
    COUNT(*) AS total_listens,
    SUM(l.duration_seconds) AS total_listen_seconds,
    ROUND(SUM(l.duration_seconds) / 3600.0, 1) AS listen_hours
FROM listens l
INNER JOIN tracks t ON l.track_id = t.track_id
INNER JOIN artists a ON t.artist_id = a.artist_id
GROUP BY a.name
ORDER BY total_listen_seconds DESC
LIMIT 10;

### COUNT(DISTINCT)

How many *different* users have listened to each genre? A plain COUNT
counts listens; `COUNT(DISTINCT ...)` counts unique values.

In [ ]:
%%sql
SELECT
    t.genre,
    COUNT(*) AS total_listens,
    COUNT(DISTINCT l.user_id) AS unique_listeners
FROM listens l
INNER JOIN tracks t ON l.track_id = t.track_id
GROUP BY t.genre
ORDER BY unique_listeners DESC;

### Subscription breakdown

How do free and premium users differ? The `1.0 *` forces floating-point
division.

In [ ]:
%%sql
SELECT
    u.subscription_type,
    COUNT(*) AS total_listens,
    COUNT(DISTINCT u.user_id) AS unique_users,
    ROUND(1.0 * COUNT(*) / COUNT(DISTINCT u.user_id), 1) AS listens_per_user
FROM listens l
INNER JOIN users u ON l.user_id = u.user_id
GROUP BY u.subscription_type;

## Exercises

### Exercise 1

How many users are premium vs free? Write a single query using GROUP BY.

### Exercise 2

Find the average, minimum, and maximum track duration per genre. Round the
average to one decimal place. Order by average duration descending.

### Exercise 3

Which artists have more than 30 tracks? Show the artist name and track
count, sorted descending. You need a JOIN and HAVING.

### Exercise 4

For each country in `users`, count total users and premium users. Order by
total descending.

**Hint:** `SUM(CASE WHEN subscription_type = 'premium' THEN 1 ELSE 0 END)`.

### Exercise 5

Find the top 5 tracks by total listening time (sum of `duration_seconds`
from `listens`). Show the track title, artist name, and total hours.

## Summary

- **COUNT, SUM, AVG, MIN, MAX** collapse rows into summary values
- **COUNT(*)** counts all rows; **COUNT(column)** excludes NULLs;
  **COUNT(DISTINCT column)** counts unique values
- **GROUP BY** splits rows into groups; every non-aggregated column in
  SELECT must appear in GROUP BY
- **WHERE** filters rows before grouping; **HAVING** filters groups after
- **COALESCE** replaces NULLs with a default value
- Combining aggregation with JOINs is where the real analytical power lives

In the next notebook, we tackle subqueries and CTEs -- the tools you need
when a single query is not enough.